# Copernicus Marine — Sea Water Salinity, Gulf of Tunisia

**Product:** `MEDSEA_MULTIYEAR_PHY_006_004`  
**Dataset:** `cmems_mod_med_phy-sal_my_4.2km_P1D-m`  
**Variable:** `so` — sea water practical salinity (PSU) at 1 m depth  
**Period:** 2026-01-01 → 2026-03-31  
**Domain:** Gulf of Tunisia — lon 10°–10.9°E, lat 36.6°–37.2°N  
**Model:** Mediterranean Sea Physics Reanalysis (MFS E3R1I, 4.2 km, daily mean)

In [ ]:
%pip install copernicusmarine hvplot matplotlib xarray --quiet

In [ ]:
import copernicusmarine

# Saves credentials to ~/.copernicusmarine so you are not prompted again
copernicusmarine.login(
    username="mazzabi1",
    password="YOUR_PASSWORD_HERE",  # <-- replace with your Copernicus Marine password
    force_overwrite=True,
)

## 1. Open the salinity dataset (lazy, no download yet)

In [ ]:
import warnings
warnings.filterwarnings("ignore")

ds_sal = copernicusmarine.open_dataset(
    dataset_id="cmems_mod_med_phy-sal_my_4.2km_P1D-m",
    variables=["so"],
    minimum_longitude=10.0,
    maximum_longitude=10.9,
    minimum_latitude=36.6,
    maximum_latitude=37.2,
    start_datetime="2026-01-01T00:00:00",
    end_datetime="2026-03-31T00:00:00",
)
ds_sal

## 2. Inspect the dataset

In [ ]:
print("Variables :", list(ds_sal.data_vars))
print("Dimensions:", dict(ds_sal.sizes))
print("Time range:", str(ds_sal.time.values[0])[:10], "->", str(ds_sal.time.values[-1])[:10])
print("Depth levels:", ds_sal.depth.values)

## 3. Select surface level (1 m depth) and load into memory

In [ ]:
%%time
so = ds_sal["so"].isel(depth=0).load()

print(f"Salinity min : {float(so.min()):.2f} PSU")
print(f"Salinity max : {float(so.max()):.2f} PSU")
print(f"Salinity mean: {float(so.mean()):.2f} PSU")

## 4. Spatial map — salinity on the first day (2026-01-01)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(11, 4.5))
so.isel(time=0).plot(ax=ax, cmap="Blues_r", cbar_kwargs={"label": "PSU"})
ax.set_title(f"Surface salinity (so, 1 m) — {str(so.time.values[0])[:10]}")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.tight_layout()
plt.show()

## 5. Spatial map — salinity on the last day (2026-03-31)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
so.isel(time=-1).plot(ax=ax, cmap="Blues_r", cbar_kwargs={"label": "PSU"})
ax.set_title(f"Surface salinity (so, 1 m) — {str(so.time.values[-1])[:10]}")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.tight_layout()
plt.show()

## 6. Temporal mean map (Jan–Mar 2026)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
so.mean(dim="time").plot(ax=ax, cmap="Blues_r", cbar_kwargs={"label": "PSU"})
ax.set_title("Mean surface salinity — Gulf of Tunisia, Jan–Mar 2026")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.tight_layout()
plt.show()

## 7. Basin-averaged daily salinity time series

In [ ]:
ts_sal = so.mean(dim=["latitude", "longitude"])

fig, ax = plt.subplots(figsize=(12, 4))
ts_sal.plot(ax=ax, color="steelblue", linewidth=1.5)
ax.set_ylabel("Salinity (PSU)")
ax.set_xlabel("Date")
ax.set_title("Basin-averaged daily surface salinity — Gulf of Tunisia, Jan–Mar 2026")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Spatial variability — standard deviation over time

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
so.std(dim="time").plot(ax=ax, cmap="OrRd", cbar_kwargs={"label": "PSU"})
ax.set_title("Salinity standard deviation — Gulf of Tunisia, Jan–Mar 2026")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.tight_layout()
plt.show()

## 9. Interactive map with time slider (hvplot)

In [ ]:
import hvplot.xarray  # noqa: F401
import panel as pn
pn.extension()

so.hvplot(
    x="longitude", y="latitude",
    groupby="time",
    rasterize=True,
    geo=True,
    tiles="OSM",
    cmap="Blues_r",
    clabel="PSU",
    clim=(float(so.min()), float(so.max())),
    title="Surface salinity (so, 1 m) — Gulf of Tunisia",
    width=700,
    height=500,
)

## 10. (Optional) Download the data as a local NetCDF file

In [ ]:
output_file = "cmems_salinity_tunisia_2026-Q1.nc"
so.to_netcdf(output_file)
print(f"Saved → {output_file}")